# 08 — SEC + caliper × video-log × Ardaman panel

Three-column convergence panel:

- **Column 1**: BP labels (just the depths, in their own column for clarity)
- **Column 2**: caliper trace + per-sample severity bands + companion
  caliper traces, with the smoothed SEC profile overlaid via a twin
  x-axis (top = SEC, bottom = caliper). BP markers on the SEC curve.
- **Column 3**: video-log notes + Ardaman 2009 (when applicable),
  *without* severity bands (notes are PAV-displaced from their anchor
  depth, so bands here would mislead the reader).

This panel is the workhorse for the SEC × caliper × video convergence
analysis. By rendering N=1..10 for both savgol and lowess, you can
visually sweep through the breakpoint hypothesis space and identify
where the techniques agree on cavity locations.

## Convention

Y-axis is **BGL-positive** (0 at top, depth increases downward),
shared with caliper, videolog, drilling and SEC. The y-title is
rendered as figure text on the leftmost edge — the in-axis ylabel
slot is reserved as a buffer between BP labels and the panel frame.

## Output

One PNG per (well, smoothing, n) combination. With 5 wells × 2
smoothings × 10 N values, a full batch produces 100 PNGs in
``results/figures/convergence/sec_caliper_video/<well>/``.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

from karst_analysis.convergence import (
    WELLS, SecCaliperVideoConfig,
    plot_sec_caliper_video_panel,
    build_all_sec_caliper_video_panels,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Wells available:")
for w, cfg in WELLS.items():
    print(f"  {w:8s}  video sheet={cfg.video_sheet:8s}  has_ardaman={cfg.has_ardaman}")

## 1. Render a single panel for inspection

In [ ]:
fig = plot_sec_caliper_video_panel(
    "LRS70D", smoothing="savgol", n=3,
    output_path="results/figures/convergence/sec_caliper_video/LRS70D/LRS70D_demo_savgol_N03.png",
)
plt.close(fig)

from IPython.display import Image
Image(filename="results/figures/convergence/sec_caliper_video/LRS70D/LRS70D_demo_savgol_N03.png")

## 2. Stress-test: N=10 for one well

This is the densest case. If the layout holds here, it holds everywhere.

In [ ]:
fig = plot_sec_caliper_video_panel(
    "LRS70D", smoothing="savgol", n=10,
    output_path="/tmp/LRS70D_stress_savgol_N10.png",
)
plt.close(fig)
Image(filename="/tmp/LRS70D_stress_savgol_N10.png")

## 3. Compare savgol vs lowess for the same N

In [ ]:
for smoothing in ("savgol", "lowess"):
    fig = plot_sec_caliper_video_panel(
        "LRS70D", smoothing=smoothing, n=5,
        output_path=f"/tmp/LRS70D_compare_{smoothing}_N05.png",
    )
    plt.close(fig)
print("Saved both. Open them side by side to compare.")

## 4. Batch: render all (wells, smoothings, N) combinations

This is what ``scripts/sec_caliper_video_panels.py`` runs internally.

For 5 wells × 2 smoothings × 10 N = up to 100 panels. Combinations
where the BIC sweep doesn't include a particular N are skipped with a
warning.

In [ ]:
# Smaller batch for the demo: just LRS70D, both smoothings, all N.
paths = build_all_sec_caliper_video_panels(
    wells=["LRS70D"],
    n_min=1, n_max=10,
    output_dir="results/figures/convergence/sec_caliper_video",
)
print(f"\n{len(paths)} panels in results/figures/convergence/sec_caliper_video/LRS70D/")

## 5. Tweaking visuals

The ``SecCaliperVideoConfig`` dataclass exposes layout parameters.

In [ ]:
# Example: bigger fonts, narrower note wrap, taller per-row spacing.
custom_cfg = SecCaliperVideoConfig(
    note_wrap=120,
    note_fontsize=10.0,
    bp_fontsize=10.0,
    auto_height_factor=0.40,
)

fig = plot_sec_caliper_video_panel(
    "LRS70D", smoothing="savgol", n=3,
    config=custom_cfg,
    output_path="/tmp/LRS70D_custom.png",
)
plt.close(fig)
Image(filename="/tmp/LRS70D_custom.png")